In [26]:
import mysql.connector
import pandas as pd
import os
from dateutil.relativedelta import relativedelta

In [2]:
class mysql_db_helper:
    def __init__(self,data = 'ecomm'):
        database_name = data
        self.create_connection({"database_name":database_name})

    def create_connection(self, data):
        try:
            # refer to understand the connection https://www.red-gate.com/simple-talk/databases/mysql/retrieving-mysql-data-python/
            self.conn = mysql.connector.connect(option_files = '/home/de24/config/mysqldbconnectors.cnf')
            self.conn.database = data['database_name']
            #self.conn.autocommit = True
            self.curr = self.conn.cursor()
        except Exception as e:
            raise Exception(f"Error => {e}")
            

    def query_exec(self,query):
        try:
            # One Query at a time
            self.curr.execute(query)          
            # 1. Check success
            print(f"Rows affected: {self.curr.rowcount}")
            # 2. Save the changes
            self.curr.commit() 
            print("Changes committed to the database.")
        except mysql.connector.Error as err:
            # 3. If an error occurs, the execution stops and goes here
            self.conn.rollback()
            # print(f"Error => {err}")
            raise Exception(f"Error => {err}")

    def query_exec_getresult(self,query):
        try:
            result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame
        except Exception as e:
            #raise Exception(f"Error => {e}")
            print(f"Error => {e}")
            return -1
        return result_df

    def connection_close(self):
        self.conn.close()

    def __del__(self):
        self.conn.close() 

In [16]:
def get_metadata_mysql(data):
    result = {}
    mysql_obj = mysql_db_helper('ecomm')
    table_name = data.get('table_name',None)
    df = mysql_obj.query_exec_getresult(f'''SELECT * FROM metadata_config  
                                            WHERE  table_name = '{table_name}';''')
    mysql_obj.connection_close()
    for column in df.columns:
        result[column] = df[column][0]

    return result 

In [21]:
main_data = {
        'source_database_name' : 'ecomm',
        'table_name' : 'customer',
        'destination_bucket' : '/home',
        'destination_s3_dir_path' : '/de24/S3_BUCKET/RAW/'
       }

meta_data = get_metadata_mysql(main_data)

/tmp/ipykernel_2303/3328980543.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


In [22]:
meta_data

{'table_name': 'customer',
 'last_extracxt_date': Timestamp('2016-08-01 00:00:00'),
 'tbl_query': 'SELECT * FROM ecomm.customer \nWHERE customer_id in (\n    SELECT customer_id \n    FROM orders  \n    WHERE YEAR(order_purchase_timestamp) = __YEAR__ \n    AND MONTH(order_purchase_timestamp) = __MONTH__\n);',
 'duration': np.int64(1),
 'duration_type': 'Month'}

In [41]:
data = {
    "table_name":main_data["table_name"],
    "source_database_name": main_data["source_database_name"],
    "destination_bucket":main_data["destination_bucket"],
    "destination_s3_dir_path":main_data["destination_s3_dir_path"],
    "meta_data":meta_data
       }

In [42]:
meta_data = data.get("meta_data",None)
last_extract_date = meta_data.get("last_extracxt_date",None)
next_last_extract_date = last_extract_date + relativedelta(months=1)
next_last_extract_date

Timestamp('2016-09-01 00:00:00')

In [43]:
def get_next_last_extract_date(data):
    meta_data = data.get("meta_data",None)
    last_extract_date = meta_data.get("last_extracxt_date",None)
    next_last_extract_date = last_extract_date + relativedelta(months=1)
    return next_last_extract_date

In [49]:
table_name = data.get('table_name')
source_database_name = data.get('source_database_name')
extract_date = get_next_last_extract_date(data)
tbl_query = data["meta_data"].get('tbl_query',None)
tbl_query = (tbl_query.replace('__YEAR__',f"'{extract_date.year}'")).replace('__MONTH__',f"'{extract_date.month}'")
print(tbl_query)

SELECT * FROM ecomm.customer 
WHERE customer_id in (
    SELECT customer_id 
    FROM orders  
    WHERE YEAR(order_purchase_timestamp) = '2016' 
    AND MONTH(order_purchase_timestamp) = '9'
);


In [50]:
folder_path = f"/home/de24/S3_BUCKET/RAW/{source_database_name}/{table_name}/{extract_date.year}{extract_date.month}/"
os.makedirs(folder_path, exist_ok=True)

mysql_obj = mysql_db_helper('ecomm')
df = mysql_obj.query_exec_getresult(f'''{tbl_query}''')

df.to_csv(f"{folder_path}/data.csv", index=False)
mysql_obj.connection_close()
df.head()

/tmp/ipykernel_2303/3328980543.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,86dc2ffce2dfff336de2f386a786e574,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP
1,683c54fc24d40ee9f8a6fc179fd9856c,4854e9b3feff728c13ee5fc7d1547e92,99025,passo fundo,RS
2,08c5351a6aca1c1589a38f244edeee9d,b7d76e111c89f7ebf14761390f0f7d17,69309,boa vista,RR
3,622e13439d6b5a0b486c435618b2679e,009b0127b727ab0ba422f6d9604487c7,12244,sao jose dos campos,SP


In [45]:
data

{'table_name': 'customer',
 'source_database_name': 'ecomm',
 'destination_bucket': '/home',
 'destination_s3_dir_path': '/de24/S3_BUCKET/RAW/',
 'meta_data': {'table_name': 'customer',
  'last_extracxt_date': Timestamp('2016-08-01 00:00:00'),
  'tbl_query': 'SELECT * FROM ecomm.customer \nWHERE customer_id in (\n    SELECT customer_id \n    FROM orders  \n    WHERE YEAR(order_purchase_timestamp) = __YEAR__ \n    AND MONTH(order_purchase_timestamp) = __MONTH__\n);',
  'duration': np.int64(1),
  'duration_type': 'Month'}}

In [55]:

    
def Upload_data_to_s3(data):
    table_name = data.get('table_name')
    extract_date = get_next_last_extract_date(data)
    tbl_query = data["meta_data"].get('tbl_query',None)
    tbl_query = (tbl_query.replace('__YEAR__',f"'{extract_date.year}'")).replace('__MONTH__',f"'{extract_date.month}'")
    
    folder_path = f"/home/de24/S3_BUCKET/RAW/{source_database_name}/{table_name}/{extract_date.year}{extract_date.month if extract_date.month > 9 else '0'+str(extract_date.month)}/"
    os.makedirs(folder_path, exist_ok=True)
    
    mysql_obj = mysql_db_helper('ecomm')
    df = mysql_obj.query_exec_getresult(f'''{tbl_query}''')
    
    df.to_csv(f"{folder_path}/data.csv", index=False)
    mysql_obj.connection_close()
    


In [56]:
Upload_data_to_s3(data)

/tmp/ipykernel_2303/3328980543.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(query, self.conn) # reading in Pandas Data frame
